In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("JSONHandling").getOrCreate()

In [6]:
from pyspark.sql.functions import *

In [3]:
customers_df = spark.read.option("multiline", "true").json("cust.json")

customers_df.printSchema()
customers_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |


In [4]:
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [7]:
flat_df = customers_df.select(
    "customer_id",
    "name",
    "city",
    "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)

flat_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [8]:
flat_df.select("name","city","phone","email").show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [9]:
flat_df.filter(col("city").isNull()).show()

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [10]:
flat_df.filter(col("phone").isNull()).show()

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|          2| Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [11]:
flat_df.filter(col("email").isNull()).show()

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|       name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|          3|John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5| Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [12]:
flat_df.filter(col("membership").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [13]:
flat_df.filter(col("preferred_service").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [14]:
flat_df.filter(col("budget_range").isNull()).show()

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|     name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|          2|Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [15]:
flat_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flat_df.columns
]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [17]:
df1 = flat_df.fillna({"city":"Unknown"})
df1.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [18]:
df2 = df1.fillna({"membership":"Standard"})
df2.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [19]:
df3 = df2.fillna({"phone":"Not Provided"})
df3.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|           NULL|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [20]:
df4 = df3.fillna({"email":"Not Provided"})
df4.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [21]:
df5 = df4.fillna({"preferred_service":"Not Selected"})
df5.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [22]:
clean_df = df5.fillna({"budget_range":"Unknown"})
clean_df.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [23]:
status_df = flat_df.withColumn(
    "customer_quality_status",
    when(
        col("city").isNull() |
        col("phone").isNull() |
        col("email").isNull() |
        col("membership").isNull() |
        col("preferred_service").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

status_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Fligh

In [24]:
status_df.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [25]:
status_df.filter(
    col("customer_quality_status")=="Complete"
).show()

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|     phone|         email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|          1|Aarav Mehta|Hyderabad|      Gold|9876500011|aarav@mail.com|           Flight|      Medium|               Complete|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [26]:
status_df.filter(
    col("customer_quality_status")=="Incomplete"
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|             Incomplete|
+-----------+------------+---------+----------+----------+---------------+----------------

In [27]:
clean_df.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [28]:
clean_df.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [29]:
flat_df.write.mode("overwrite").parquet("customers_flat.parquet")

In [30]:
clean_df.write.mode("overwrite").option("header",True).csv("clean_customers.csv")

In [31]:
print("Original Count =", flat_df.count())
print("Clean Count =", clean_df.count())

Original Count = 5
Clean Count = 5


In [32]:
flat_df.filter(
    col("phone").isNull() |
    col("email").isNull()
).show()

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|       name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|          2|  Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5| Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [33]:
flat_df.filter(
    col("preferred_service").isNull() |
    col("budget_range").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

